In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names are loaded from 00_project_config.
# Reusable validation, cleaning, deduplication, and writing helpers are loaded
# from 01_spark_utils.

BRONZE_TABLE = f"{BRONZE_SCHEMA}.api_team_data"
SILVER_TABLE = f"{SILVER_SCHEMA}.team_data"

# One Silver row represents a team participating in a competition season.
BUSINESS_KEY = [
    "team_id",
    "league_id",
    "season",
]

REQUIRED_BRONZE_COLUMNS = [
    "data",
    "league_id",
    "league_name",
    "season",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

REQUIRED_SILVER_COLUMNS = [
    "team_id",
    "team_name",
    "league_id",
    "league_name",
    "season",
]


# -----------------------------------------------------------------------------
# Validate Bronze input
# -----------------------------------------------------------------------------
# A missing Bronze source is treated as a pipeline failure rather than a
# warning, because continuing could produce a stale or incomplete Silver table.

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Required Bronze table does not exist: {BRONZE_TABLE}"
    )


# -----------------------------------------------------------------------------
# Read Bronze data
# -----------------------------------------------------------------------------
# Bronze preserves the nested API-Football structure:
#
#   data.team
#   data.venue
#
# along with league and season context.

bronze_df = spark.table(BRONZE_TABLE)

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Flatten, clean, and cast Silver columns
# -----------------------------------------------------------------------------
# Nested API-Football team and venue fields are converted into a flat,
# analytical Silver schema.
#
# clean_string() trims strings and converts empty strings to NULL.
#
# Source IDs remain numeric because API-Football exposes them as stable numeric
# identifiers.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        F.col("data.team.id")
        .cast("long")
        .alias("team_id"),

        clean_string("data.team.name")
        .alias("team_name"),

        clean_string("data.team.code")
        .alias("team_code"),

        clean_string("data.team.country")
        .alias("team_country"),

        F.col("data.team.founded")
        .cast("int")
        .alias("team_founded"),

        F.col("data.team.national")
        .cast("boolean")
        .alias("team_national"),


        # ---------------------------------------------------------------------
        # Venue
        # ---------------------------------------------------------------------

        F.col("data.venue.id")
        .cast("long")
        .alias("venue_id"),

        clean_string("data.venue.name")
        .alias("venue_name"),

        clean_string("data.venue.address")
        .alias("venue_address"),

        clean_string("data.venue.city")
        .alias("venue_city"),

        F.col("data.venue.capacity")
        .cast("int")
        .alias("venue_capacity"),

        clean_string("data.venue.surface")
        .alias("venue_surface"),


        # ---------------------------------------------------------------------
        # Competition / season context
        # ---------------------------------------------------------------------

        F.col("league_id")
        .cast("long")
        .alias("league_id"),

        clean_string("league_name")
        .alias("league_name"),

        F.col("season")
        .cast("int")
        .alias("season"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Future entity standardization placeholders
# -----------------------------------------------------------------------------
# The next project phase will standardize entity names and identifiers across
# API-Football, FBref, and football-data.co.uk.
#
# Important candidates from this dataset include:
#
#   team_name
#   team_country
#   venue_name
#   venue_address
#   venue_city
#   league_name
#
# The preferred future approach is to preserve source values and introduce
# canonical platform identifiers / standardized attributes rather than
# overwriting the source-derived fields.
#
# Example future model:
#
#   team_name_source
#   team_name_standardized
#   team_key
#
#   venue_name_source
#   venue_name_standardized
#   venue_key
#
#   competition_key
#
# No physical placeholder columns are added yet because NULL-only or duplicate
# columns would add schema noise before the standardization logic exists.


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Every row must identify a specific team within a competition season.
#
# Validation happens before deduplication so invalid NULL keys cannot be grouped
# together and accidentally reduced to a single record.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so repeated loads of the same raw source
# can create multiple versions of the same team/league/season record.
#
# Keep the version from the latest Bronze ingestion event.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# These columns are required for a usable team record.
#
# The helper also provides the final row count, avoiding a separate count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team-season records into {SILVER_TABLE}."
)